## 1、Import the required
thon libraries for Apache Spark and Cassandra integration.
In this section, the required Python libraries are imported for Apache Spark processing and Cassandra integration. SparkSession is used to create the Spark application, while Spark SQL functions and data types are used for data transformation and schema definition.

In [1]:
# Import system libraries
import os
import sys

# Set Python executable for local PySpark execution on Windows
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Import PySpark libraries
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.types import StringType

print("Python executable:", python_path)
print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)

Python executable: D:\anaconda\python.exe
Python version: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
PySpark version: 4.1.2


## 2、load data into HDFS

In this section, the MovieLens 100k dataset is downloaded, extracted, and uploaded into HDFS. The assignment requires three files from the dataset: `u.user`, `u.data`, and `u.item`.

HDFS is used as the raw data storage layer in this pipeline. After the files are uploaded to HDFS, Apache Spark will read the raw files from HDFS and transform them into RDDs and DataFrames for further processing.

```bash
# Go to the working directory
cd /home/maria_dev

# Download the MovieLens 100k dataset
wget https://files.grouplens.org/datasets/movielens/ml-100k.zip

# Extract the downloaded zip file
unzip ml-100k.zip

# Create the target directory in HDFS
hdfs dfs -mkdir -p /user/maria_dev/assignment02/movielens/raw

# Upload u.user,u.data and u.item file to HDFS
hdfs dfs -put -f /home/maria_dev/ml-100k/u.user /user/maria_dev/assignment02/movielens/raw/
hdfs dfs -put -f /home/maria_dev/ml-100k/u.data /user/maria_dev/assignment02/movielens/raw/
hdfs dfs -put -f /home/maria_dev/ml-100k/u.item /user/maria_dev/assignment02/movielens/raw/

# Check whether the files have been uploaded successfully
hdfs dfs -ls /user/maria_dev/assignment02/movielens/raw/
```

After running the commands above, the three required MovieLens files are stored in the HDFS directory `/user/maria_dev/assignment02/movielens/raw/`. This confirms that the raw dataset has been successfully loaded into the distributed file system and is ready to be processed by Apache Spark.


## 3. Create RDDs from Raw Dataset
In this section, the raw MovieLens files stored in HDFS are loaded using SparkContext. Each file is read as a Resilient Distributed Dataset (RDD). These RDDs will later be parsed and transformed into Spark DataFrames for data cleaning, preprocessing, and analysis.

The three raw input files are:
- `u.user`: user demographic information
- `u.data`: user movie rating records
- `u.item`: movie information and genre indicators

In [2]:
# Create SparkSession for local notebook execution
spark = SparkSession.builder \
    .appName("MovieLens Data Management Assignment 02") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.executorEnv.PYSPARK_PYTHON", python_path) \
    .getOrCreate()

# Get SparkContext
sc = spark.sparkContext

print("Spark version:", spark.version)
print("Spark Python executable:", sc.pythonExec)
# Choose data source mode
# "local" is used for running this notebook in the local development environment.
# "hdfs" is used when running the same logic inside the Hadoop/Spark server environment.

DATA_SOURCE = "local"

Spark version: 4.1.2
Spark Python executable: D:\anaconda\python.exe


In [3]:
# Define file paths based on the selected data source
if DATA_SOURCE == "local":
    user_path = "data/u.user"
    rating_path = "data/u.data"
    movie_path = "data/u.item"

elif DATA_SOURCE == "hdfs":
    user_path = "hdfs:///user/maria_dev/assignment02/movielens/raw/u.user"
    rating_path = "hdfs:///user/maria_dev/assignment02/movielens/raw/u.data"
    movie_path = "hdfs:///user/maria_dev/assignment02/movielens/raw/u.item"

else:
    raise ValueError("Invalid DATA_SOURCE. Please choose either 'local' or 'hdfs'.")

print("Current data source:", DATA_SOURCE)
print("User file path:", user_path)
print("Rating file path:", rating_path)
print("Movie file path:", movie_path)


# Create RDDs from raw files
user_rdd = sc.textFile(user_path)
rating_rdd = sc.textFile(rating_path)
movie_rdd = sc.textFile(movie_path)

Current data source: local
User file path: data/u.user
Rating file path: data/u.data
Movie file path: data/u.item


In [4]:
# Display the first 5 records from each RDD for verification
print("First 5 records from u.user:")
for line in user_rdd.take(5):
    print(line)

print("\nFirst 5 records from u.data:")
for line in rating_rdd.take(5):
    print(line)

print("\nFirst 5 records from u.item:")
for line in movie_rdd.take(5):
    print(line)

First 5 records from u.user:
1|24|M|technician|85711
2|53|F|other|94043
3|23|M|writer|32067
4|24|M|technician|43537
5|33|F|other|15213

First 5 records from u.data:
196	242	3	881250949
186	302	3	891717742
22	377	1	878887116
244	51	2	880606923
166	346	1	886397596

First 5 records from u.item:
1|Toy Story (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)|0|0|0|1|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0
2|GoldenEye (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?GoldenEye%20(1995)|0|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0
3|Four Rooms (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995)|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0
4|Get Shorty (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Get%20Shorty%20(1995)|0|1|0|0|0|1|0|0|1|0|0|0|0|0|0|0|0|0|0
5|Copycat (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Copycat%20(1995)|0|0|0|0|0|0|1|0|1|0|0|0|0|0|0|0|1|0|0


In [5]:
# Count the number of records in each RDD
print("Number of user records:", user_rdd.count())
print("Number of rating records:", rating_rdd.count())
print("Number of movie records:", movie_rdd.count())

Number of user records: 943
Number of rating records: 100000
Number of movie records: 1682


### HDFS RDD Loading Validation

The MovieLens raw files were also uploaded to HDFS and validated in the Hadoop/Spark server environment using `spark-submit`. The following screenshot confirms that `u.user`, `u.data`, and `u.item` were successfully loaded from HDFS into Spark RDDs.

The following script was executed in the Hadoop/Spark virtual machine environment to validate that Spark could read the three MovieLens files from HDFS and create RDDs successfully:
```bash
spark-submit /home/maria_dev/assignment02/scripts/read_hdfs_files.py
```
<img src="screenshots/rdd_loading_from_hdfs.png" width="800">


## 4. Transform RDDs into Spark DataFrames

In this section, the raw RDDs are parsed and transformed into Spark DataFrames. Each MovieLens file has a different structure and delimiter. The `u.user` and `u.item` files are separated by the pipe symbol `|`, while the `u.data` file is separated by whitespace or tab characters.

The DataFrames created in this section will be used for data cleaning, preprocessing, Cassandra storage, MongoDB storage, and analytical queries.

In [6]:
# ==========================================================
# 4. Transform RDDs into Spark DataFrames
# ==========================================================

# ----------------------------------------------------------
# 4.1 Parse u.user into users_df
# ----------------------------------------------------------

# u.user format:
# user_id | age | gender | occupation | zip_code

def parse_user(line):
    fields = line.split("|")

    return Row(
        user_id=int(fields[0]),
        age=int(fields[1]),
        gender=fields[2],
        occupation=fields[3],
        zip_code=fields[4]
    )


# Convert user RDD to DataFrame
users_df = spark.createDataFrame(user_rdd.map(parse_user))

print("Users DataFrame Schema:")
users_df.printSchema()

print("Sample records from users_df:")
users_df.show(5, truncate=False)

Users DataFrame Schema:
root
 |-- user_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- zip_code: string (nullable = true)

Sample records from users_df:
+-------+---+------+----------+--------+
|user_id|age|gender|occupation|zip_code|
+-------+---+------+----------+--------+
|1      |24 |M     |technician|85711   |
|2      |53 |F     |other     |94043   |
|3      |23 |M     |writer    |32067   |
|4      |24 |M     |technician|43537   |
|5      |33 |F     |other     |15213   |
+-------+---+------+----------+--------+
only showing top 5 rows


In [7]:
# ----------------------------------------------------------
# 4.2 Parse u.data into ratings_df
# ----------------------------------------------------------

# u.data format:
# user_id movie_id rating timestamp
# The fields are separated by tab or whitespace.

def parse_rating(line):
    fields = line.split()

    return Row(
        user_id=int(fields[0]),
        movie_id=int(fields[1]),
        rating=int(fields[2]),
        timestamp=int(fields[3])
    )


# Convert rating RDD to DataFrame
ratings_df = spark.createDataFrame(rating_rdd.map(parse_rating))

print("Ratings DataFrame Schema:")
ratings_df.printSchema()

print("Sample records from ratings_df:")
ratings_df.show(5, truncate=False)

Ratings DataFrame Schema:
root
 |-- user_id: long (nullable = true)
 |-- movie_id: long (nullable = true)
 |-- rating: long (nullable = true)
 |-- timestamp: long (nullable = true)

Sample records from ratings_df:
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|196    |242     |3     |881250949|
|186    |302     |3     |891717742|
|22     |377     |1     |878887116|
|244    |51      |2     |880606923|
|166    |346     |1     |886397596|
+-------+--------+------+---------+
only showing top 5 rows


In [8]:
# ----------------------------------------------------------
# 4.3 Parse u.item into movies_df
# ----------------------------------------------------------

# u.item format:
# movie_id | movie_title | release_date | video_release_date | imdb_url | 19 genre columns

# Define MovieLens genre columns
genre_columns = [
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film_Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci_Fi", "Thriller", "War", "Western"
]


def parse_movie(line):
    fields = line.split("|")

    movie_id = int(fields[0])
    movie_title = fields[1]
    release_date = fields[2]
    video_release_date = fields[3]
    imdb_url = fields[4]

    # Convert genre indicators from string to integer
    genre_values = [int(value) for value in fields[5:24]]

    # Create dictionary for row data
    row_data = {
        "movie_id": movie_id,
        "movie_title": movie_title,
        "release_date": release_date,
        "video_release_date": video_release_date,
        "imdb_url": imdb_url
    }

    # Add genre columns into row data
    for genre_name, genre_value in zip(genre_columns, genre_values):
        row_data[genre_name] = genre_value

    return Row(**row_data)


# Convert movie RDD to DataFrame
movies_df = spark.createDataFrame(movie_rdd.map(parse_movie))

print("Movies DataFrame Schema:")
movies_df.printSchema()

print("Sample records from movies_df:")
movies_df.show(5, truncate=False)

Movies DataFrame Schema:
root
 |-- movie_id: long (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- video_release_date: string (nullable = true)
 |-- imdb_url: string (nullable = true)
 |-- unknown: long (nullable = true)
 |-- Action: long (nullable = true)
 |-- Adventure: long (nullable = true)
 |-- Animation: long (nullable = true)
 |-- Children: long (nullable = true)
 |-- Comedy: long (nullable = true)
 |-- Crime: long (nullable = true)
 |-- Documentary: long (nullable = true)
 |-- Drama: long (nullable = true)
 |-- Fantasy: long (nullable = true)
 |-- Film_Noir: long (nullable = true)
 |-- Horror: long (nullable = true)
 |-- Musical: long (nullable = true)
 |-- Mystery: long (nullable = true)
 |-- Romance: long (nullable = true)
 |-- Sci_Fi: long (nullable = true)
 |-- Thriller: long (nullable = true)
 |-- War: long (nullable = true)
 |-- Western: long (nullable = true)

Sample records from movies_df:
+--------+----------

In [9]:
# ----------------------------------------------------------
# 4.4 Validate DataFrame Record Counts
# ----------------------------------------------------------

print("Number of users:", users_df.count())
print("Number of ratings:", ratings_df.count())
print("Number of movies:", movies_df.count())

Number of users: 943
Number of ratings: 100000
Number of movies: 1682


In [10]:
# ----------------------------------------------------------
# 4.5 Display DataFrame Columns
# ----------------------------------------------------------

print("Users DataFrame columns:")
print(users_df.columns)

print("\nRatings DataFrame columns:")
print(ratings_df.columns)

print("\nMovies DataFrame columns:")
print(movies_df.columns)

Users DataFrame columns:
['user_id', 'age', 'gender', 'occupation', 'zip_code']

Ratings DataFrame columns:
['user_id', 'movie_id', 'rating', 'timestamp']

Movies DataFrame columns:
['movie_id', 'movie_title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western']


## 5. Data Cleaning and Preprocessing
In this section, basic data cleaning and preprocessing steps are performed on the three Spark DataFrames. The purpose is to ensure that the data is valid, consistent, and ready for database storage and analytical queries.

The preprocessing includes:

- Checking missing values
- Checking duplicate records
- Validating rating and age values
- Creating a movie-genre mapping DataFrame for genre-based analysis

### 5.1 Check Missing Values

In [11]:
# ==========================================================
# 5. Data Cleaning and Preprocessing
# ==========================================================

# ----------------------------------------------------------
# 5.1 Check Missing Values
# ----------------------------------------------------------

# ----------------------------------------------------------
# Check Missing Values
# ----------------------------------------------------------

from pyspark.sql.types import StringType

def check_missing_values(df, df_name):
    print("=" * 50)
    print(f"Missing Values in {df_name}")
    print("=" * 50)

    missing_exprs = []

    for field in df.schema.fields:
        column_name = field.name
        column_type = field.dataType

        # For string columns, check both NULL and empty string
        if isinstance(column_type, StringType):
            missing_exprs.append(
                F.sum(
                    F.when(
                        F.col(column_name).isNull() | (F.trim(F.col(column_name)) == ""),
                        1
                    ).otherwise(0)
                ).alias(column_name)
            )

        # For non-string columns, only check NULL
        else:
            missing_exprs.append(
                F.sum(
                    F.when(
                        F.col(column_name).isNull(),
                        1
                    ).otherwise(0)
                ).alias(column_name)
            )

    df.select(missing_exprs).show(truncate=False)


# Check missing values in each DataFrame
check_missing_values(users_df, "users_df")
check_missing_values(ratings_df, "ratings_df")
check_missing_values(movies_df, "movies_df")

Missing Values in users_df
+-------+---+------+----------+--------+
|user_id|age|gender|occupation|zip_code|
+-------+---+------+----------+--------+
|0      |0  |0     |0         |0       |
+-------+---+------+----------+--------+

Missing Values in ratings_df
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|0      |0       |0     |0        |
+-------+--------+------+---------+

Missing Values in movies_df
+--------+-----------+------------+------------------+--------+-------+------+---------+---------+--------+------+-----+-----------+-----+-------+---------+------+-------+-------+-------+------+--------+---+-------+
|movie_id|movie_title|release_date|video_release_date|imdb_url|unknown|Action|Adventure|Animation|Children|Comedy|Crime|Documentary|Drama|Fantasy|Film_Noir|Horror|Musical|Mystery|Romance|Sci_Fi|Thriller|War|Western|
+--------+-----------+------------+------------------+--------+-------+------+---------+---------

### 5.2 Check Duplicate Records

In [12]:
# ----------------------------------------------------------
# 5.2 Check Duplicate Records
# ----------------------------------------------------------

print("Duplicate check:")

print("Users total records:", users_df.count())
print("Users distinct records:", users_df.distinct().count())

print("Ratings total records:", ratings_df.count())
print("Ratings distinct records:", ratings_df.distinct().count())

print("Movies total records:", movies_df.count())
print("Movies distinct records:", movies_df.distinct().count())

Duplicate check:
Users total records: 943
Users distinct records: 943
Ratings total records: 100000
Ratings distinct records: 100000
Movies total records: 1682
Movies distinct records: 1682


### 5.3 Create Cleaned DataFrames

In [13]:
# ----------------------------------------------------------
# 5.3 Create Cleaned DataFrames
# ----------------------------------------------------------

users_clean_df = users_df \
    .dropDuplicates(["user_id"]) \
    .filter(F.col("user_id").isNotNull()) \
    .filter((F.col("age") > 0) & (F.col("age") <= 100))

ratings_clean_df = ratings_df \
    .dropDuplicates(["user_id", "movie_id", "timestamp"]) \
    .filter(F.col("user_id").isNotNull()) \
    .filter(F.col("movie_id").isNotNull()) \
    .filter((F.col("rating") >= 1) & (F.col("rating") <= 5))

movies_clean_df = movies_df \
    .dropDuplicates(["movie_id"]) \
    .filter(F.col("movie_id").isNotNull()) \
    .filter(F.col("movie_title").isNotNull())

print("Cleaned users count:", users_clean_df.count())
print("Cleaned ratings count:", ratings_clean_df.count())
print("Cleaned movies count:", movies_clean_df.count())

Cleaned users count: 943
Cleaned ratings count: 100000
Cleaned movies count: 1682


### 5.4 Create Movie-Genre Mapping DataFrame
The original `u.item` file contains movie information and 19 binary genre indicator columns. Each genre column uses `1` or `0` to indicate whether a movie belongs to that genre. However, storing genres as many separate binary columns is not very intuitive for later processing and database storage.

Therefore, during the preprocessing step, the 19 genre indicator columns were converted into a single array column called `genres`. Since one movie may belong to multiple genres, the array structure can preserve all genre information for each movie. For example, `Toy Story (1995)` can be stored with multiple genres such as `Animation`, `Children`, and `Comedy`.

This transformation makes the movie data cleaner and easier to understand. It also makes the `u.item` dataset more suitable for MongoDB storage. Compared with the `u.data` rating dataset, which contains numerical rating records and is more suitable for structured storage and aggregation, the movie dataset mainly contains descriptive information such as movie title, release date, IMDb URL, and genre list. These fields fit naturally into a document-style structure.

In this project, the processed movie data will be stored in MongoDB, where each movie can be represented as one document. The `genres` field can be stored as an array inside the document, which is more flexible than storing 19 separate genre columns. This design also supports later analysis, because the `genres` array can be temporarily exploded in Spark when genre-level aggregation is required.

In [14]:
# ----------------------------------------------------------
# Create movies_mongo_df with genres as an array
# ----------------------------------------------------------

from pyspark.sql import Row
from pyspark.sql.types import *

# MovieLens 100k genre names
genre_columns = [
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film_Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci_Fi", "Thriller", "War", "Western"
]

def parse_movie_for_mongodb(line):
    fields = line.split("|")

    movie_id = int(fields[0])
    movie_title = fields[1]
    release_date = fields[2]
    video_release_date = fields[3]
    imdb_url = fields[4]

    # Convert 19 binary genre indicators into a genre list
    genres = []
    genre_values = fields[5:24]

    for genre_name, genre_value in zip(genre_columns, genre_values):
        if int(genre_value) == 1:
            genres.append(genre_name)

    return Row(
        movie_id=movie_id,
        movie_title=movie_title,
        release_date=release_date,
        video_release_date=video_release_date,
        imdb_url=imdb_url,
        genres=genres
    )

# Convert movie RDD to MongoDB-style DataFrame
movies_mongo_df = spark.createDataFrame(movie_rdd.map(parse_movie_for_mongodb))

# Display schema and sample records
movies_mongo_df.printSchema()
movies_mongo_df.show(10, truncate=False)

root
 |-- movie_id: long (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- video_release_date: string (nullable = true)
 |-- imdb_url: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)

+--------+----------------------------------------------------+------------+------------------+--------------------------------------------------------------+-----------------------------+
|movie_id|movie_title                                         |release_date|video_release_date|imdb_url                                                      |genres                       |
+--------+----------------------------------------------------+------------+------------------+--------------------------------------------------------------+-----------------------------+
|1       |Toy Story (1995)                                    |01-Jan-1995 |                  |http://us.imdb.com/M/title-exact

## 6. Database Storage Design

Before writing the processed DataFrames into databases, the database storage design was considered based on the structure and characteristics of the three MovieLens 100k dataset files: `u.user`, `u.data`, and `u.item`.

In this project, Cassandra and MongoDB are used for different types of data storage. Cassandra is used as the main structured NoSQL database for storing user data, rating data, and final analytical result tables. MongoDB is used as an optional document-oriented database for storing semi-structured movie information with multiple genres.

The overall database design is shown below:

| Dataset / Result | Processed DataFrame | Target Database | Reason |
|---|---|---|---|
| `u.user` | `users_clean_df` | Cassandra | Structured user profile data with a fixed schema |
| `u.data` | `ratings_clean_df` | Cassandra | Large structured rating records suitable for tabular storage |
| `u.item` | `movies_mongo_df` | MongoDB | Movie information with multiple genres stored as an array |
| Task 1 result | `average_movie_ratings_df` | Cassandra | Analytical result table for movie average ratings |
| Task 2 result | `top_10_movies_df` | Cassandra | Analytical result table for top-rated movies |
| Task 3 result | `favourite_genre_df` | Cassandra | Analytical result table for active users' favourite genres |
| Task 4 result | CQL query result | Cassandra | Filtered user result table for users younger than 20 |
| Task 5 result | CQL query result | Cassandra | Filtered user result table for scientist users aged 30 to 40 |

---

### 6.1 Cassandra for Structured Data and Analytical Results

Cassandra is used as the main structured NoSQL database in this project.

First, Cassandra stores the `users` and `ratings` datasets because these two datasets are highly structured and have fixed schemas.

The `u.user` dataset contains user demographic information, including user ID, age, gender, occupation, and zip code. Each user has one record, and the schema is simple and consistent.

The `u.data` dataset contains movie rating records, including user ID, movie ID, rating, and timestamp. This dataset has a large number of records and works like a fact table in the analytical pipeline.

Therefore, the following cleaned DataFrames are stored in Cassandra as base tables:

- `users_clean_df`
- `ratings_clean_df`

Second, after the analytical tasks are completed, the final result DataFrames are also stored back into Cassandra as result tables. This makes the pipeline easier to validate and allows the analysis results to be retrieved directly from Cassandra.

The planned Cassandra base tables are:

| Cassandra Table | Source DataFrame | Main Fields |
|---|---|---|
| `users` | `users_clean_df` | `user_id`, `age`, `gender`, `occupation`, `zip_code` |
| `ratings` | `ratings_clean_df` | `user_id`, `movie_id`, `rating`, `timestamp` |

The planned Cassandra analytical result tables are:

| Cassandra Result Table | Source Result | Main Fields |
|---|---|---|
| `task1_average_movie_ratings` | Task 1 result | `movie_id`, `movie_title`, `average_rating`, `rating_count` |
| `task2_top_10_movies` | Task 2 result | `rank`, `movie_id`, `movie_title`, `average_rating`, `rating_count` |
| `task3_user_favourite_genres` | Task 3 result | `user_id`, `favourite_genre`, `genre_rating_count` |
| `task4_young_users` | Task 4 result | `user_id`, `age`, `gender`, `occupation`, `zip_code` |
| `task5_scientist_users` | Task 5 result | `user_id`, `age`, `gender`, `occupation`, `zip_code` |

Cassandra is selected because it is suitable for structured tabular data, distributed storage, and fast read/write operations on large-scale records. Storing the final analytical results in Cassandra also keeps the final output layer simple and consistent with the main database requirement of the assignment.

---

### 6.2 MongoDB for Semi-structured Movie Data

MongoDB is used to store the movie dataset derived from `u.item`.

The original `u.item` file contains basic movie information and 19 binary genre indicator columns. Each genre column uses `1` or `0` to indicate whether a movie belongs to that genre. However, this structure is not very intuitive for document-based storage, especially because one movie may belong to multiple genres.

Therefore, during preprocessing, the 19 binary genre columns were converted into a single array field called `genres`.

For example, a movie can be represented as the following document:

```json
{
  "movie_id": 1,
  "movie_title": "Toy Story (1995)",
  "release_date": "01-Jan-1995",
  "video_release_date": "",
  "imdb_url": "http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)",
  "genres": ["Animation", "Children", "Comedy"]
}
```

This structure is more suitable for MongoDB because MongoDB supports flexible document structures and array fields. Compared with the rating dataset, the movie dataset mainly contains descriptive information such as movie title, release date, IMDb URL, and a list of genres. These fields naturally fit into a document-oriented database.

Therefore, the following DataFrame is stored in MongoDB:

- `movies_mongo_df`

The planned MongoDB collection is:

| MongoDB Database | Collection | Source DataFrame | Main Fields |
|---|---|---|---|
| `movielens_db` | `movies` | `movies_mongo_df` | `movie_id`, `movie_title`, `release_date`, `imdb_url`, `genres` |

---

### 6.3 Why Different Databases Are Used

Different databases are used because the datasets have different structures and purposes.

Cassandra is more suitable for `u.user` and `u.data` because both datasets are structured and table-like. The fields are fixed, and the data can be stored in rows and columns.

MongoDB is more suitable for `u.item` because movie information is more descriptive and semi-structured. In particular, the `genres` field can naturally be stored as an array inside a MongoDB document.

This design also makes the later analytical process clearer. Spark can read structured user and rating data from Cassandra, read movie documents from MongoDB, and then join them together for analytical tasks.

After the analytical tasks are completed, the final results are stored back into Cassandra. This avoids adding an additional database layer and keeps the pipeline simpler and more stable in the current virtual machine environment.

The database role summary is:

```text
Cassandra:
- Store structured user data
- Store structured rating records
- Store final analytical result tables

MongoDB:
- Store movie documents
- Store multiple genres as an array

Spark:
- Read data from Cassandra and MongoDB
- Perform joins, aggregations, ranking, and genre-level analysis

CQL:
- Perform simple filtering tasks on Cassandra users table
- Validate Cassandra base tables and result tables
```

---

### 6.4 Database Service Validation

Before storing data into Cassandra and MongoDB, both database services were checked in the virtual machine environment.

The following commands were used to check Cassandra:

```bash
# Check Cassandra service status
service cassandra status

# Enter Cassandra CQL shell
cqlsh

# Show existing keyspaces
DESCRIBE KEYSPACES;
```

If Cassandra is running successfully, the `cqlsh` command can enter the Cassandra shell and display the available keyspaces.

#### Cassandra Service Validation

The screenshot below shows that Cassandra was successfully accessed in the virtual machine environment.

<img src="screenshots/cassandra_status.png" width="800">

---

The following commands were used to check MongoDB:

```bash
# Check MongoDB service status
service mongod status

# Enter MongoDB shell
mongo

# Show existing databases
show dbs
```

If the `mongo` shell can be opened and the `show dbs` command returns database information, MongoDB is running successfully.

#### MongoDB Service Validation

The screenshot below shows that MongoDB was successfully accessed in the virtual machine environment.

<img src="screenshots/mongodb_status.png" width="800">

---

### 6.5 Data Flow after Database Storage

After the processed data is stored in Cassandra and MongoDB, the next step is to read the data back for validation and analysis.

The planned data flow is:

```text
Cleaned Spark DataFrames
        ↓
Cassandra: users and ratings
MongoDB: movies with genres array
        ↓
Read data back into Spark
        ↓
Perform analytical tasks using Spark SQL and CQL
        ↓
Store final analytical results back into Cassandra
        ↓
Read Cassandra result tables for validation and reporting
```

This design separates the roles of different databases while keeping the final result storage simple and stable. Cassandra is used as both the main structured storage layer and the final analytical result storage layer, while MongoDB is used for flexible movie document storage.

## 7. Store Data into Cassandra and MongoDB


In this section, the processed DataFrames are stored into two different NoSQL databases based on the database storage design.

The cleaned user and rating data are stored in Cassandra because they are structured and have fixed schemas. The processed movie data is stored in MongoDB because each movie may contain multiple genres, which can be naturally represented as an array field in a document.

The storage plan is:

| DataFrame | Target Database | Target Table / Collection |
|---|---|---|
| `users_clean_df` | Cassandra | `movielens_ks.users` |
| `ratings_clean_df` | Cassandra | `movielens_ks.ratings` |
| `movies_mongo_df` | MongoDB | `movielens_db.movies` |

The following Cassandra CQL commands were used to create the keyspace and tables:

```sql
CREATE KEYSPACE IF NOT EXISTS movielens_ks
WITH replication = {
    'class': 'SimpleStrategy',
    'replication_factor': 1
};

USE movielens_ks;

CREATE TABLE users (
    user_id int PRIMARY KEY,
    age int,
    gender text,
    occupation text,
    zip_code text
);

CREATE TABLE ratings (
    user_id int,
    movie_id int,
    rating int,
    timestamp int,
    PRIMARY KEY ((user_id), movie_id, timestamp)
);
```

MongoDB does not require a fixed schema in advance. The `movies` collection will be created automatically when movie documents are inserted.

### 7.1 Database Storage Validation

The following screenshot shows that the cleaned user and rating data were successfully inserted into Cassandra, and the processed movie documents were successfully inserted into MongoDB.

```bash
spark-submit /home/maria_dev/assignment02/scripts/store_to_databases.py
```

This script performs the following steps:

1. Reads `u.user`, `u.data`, and `u.item` from HDFS.
2. Converts the raw files into Spark RDDs.
3. Parses the RDDs into Spark DataFrames.
4. Cleans and preprocesses the DataFrames.
5. Stores `users_clean_df` and `ratings_clean_df` into Cassandra.
6. Stores `movies_mongo_df` into MongoDB.
7. Reads the stored records back from Cassandra and MongoDB for validation.

The validation result confirms that:

| Database | Table / Collection | Number of Records |
|---|---|---:|
| Cassandra | `users` | 943 |
| Cassandra | `ratings` | 100000 |
| MongoDB | `movies` | 1682 |

<img src="screenshots/store_to_cassandra_mongodb.png" width="800">

### 7.2 Database Table Validation

After inserting the processed data into Cassandra and MongoDB, the records were checked directly from each database.

#### Cassandra Validation

<img src="screenshots/cassandra_table_validation.png" width="800">

#### MongoDB Validation

<img src="screenshots/mongodb_collection_validation.png" width="800">

## 8. Analytical Tasks and Cassandra Result Storage

In this section, the data stored in Cassandra and MongoDB is read back for validation and analytical processing. Cassandra stores the structured user and rating records, while MongoDB stores movie documents with genre arrays.

Spark SQL is used for complex analytical tasks involving joins, aggregations, ranking, and genre-level analysis. CQL is used for simple user filtering tasks directly on the Cassandra `users` table.

After each analytical task is completed, the final result is stored back into Cassandra as a separate result table. This allows the analytical outputs to be persisted and validated without recomputing the full pipeline.

The analysis and result storage strategy is:

| Task                                     | Data Source                        | Method    | Cassandra Result Table        |
| ---------------------------------------- | ---------------------------------- | --------- | ----------------------------- |
| Task 1: Average rating for each movie    | Cassandra ratings + MongoDB movies | Spark SQL | `task1_average_movie_ratings` |
| Task 2: Top 10 movies by average rating  | Task 1 result                      | Spark SQL | `task2_top_10_movies`         |
| Task 3: Favourite genre for active users | Cassandra ratings + MongoDB movies | Spark SQL | `task3_user_favourite_genres` |
| Task 4: Users younger than 20            | Cassandra users                    | CQL       | `task4_young_users`           |
| Task 5: Scientists aged 30 to 40         | Cassandra users                    | CQL       | `task5_scientist_users`       |

The Cassandra result tables were created before running the analytical scripts. These tables are used to store the final outputs of each task and support result validation.

---

### 8.1 Task 1 and Task 2: Movie Rating Analysis

Task 1 calculates the average rating for each movie. The result includes `movie_id`, `movie_title`, `average_rating`, and `rating_count`.

Task 2 identifies the top 10 movies with the highest average ratings. Since some movies may have very high average ratings because they received only a small number of ratings, the `rating_count` field is also included to provide additional context.

The analysis process is:

```text
Cassandra ratings
        +
MongoDB movies
        ↓
Spark SQL join by movie_id
        ↓
Calculate average rating and rating count
        ↓
Store Task 1 result into Cassandra
        ↓
Rank movies by average rating
        ↓
Store Task 2 result into Cassandra
```

The following script was executed in the Hadoop/Spark virtual machine environment:

```bash
spark-submit /home/maria_dev/assignment02/scripts/task_1_2_movie_ratings.py
```

The Task 1 result is stored in Cassandra table:

```text
task1_average_movie_ratings
```

The Task 2 result is stored in Cassandra table:

```text
task2_top_10_movies
```

The following screenshot shows that Task 1 and Task 2 were successfully completed and the results were generated.

<img src="screenshots/analysis_task_1_2.png" width="800">

The second screenshot shows that the Task 1 and Task 2 results were successfully written back into Cassandra and validated from the result tables.

<img src="screenshots/task_1_2_cassandra_storage.png" width="800">

---

### 8.2 Task 3: Active Users and Favourite Movie Genre

Task 3 identifies users who rated at least 50 movies and determines their favourite movie genre based on the genre they rated most frequently.

In MongoDB, movie genres are stored as an array field. For example, one movie may have several genres such as `Animation`, `Children`, and `Comedy`. During analysis, Spark temporarily explodes the `genres` array into multiple rows for genre-level aggregation. This does not change the MongoDB storage structure; it is only used for analytical processing.

The analysis process is:

```text
Cassandra ratings
        +
MongoDB movies with genres array
        ↓
Explode genres array in Spark
        ↓
Find users who rated at least 50 movies
        ↓
Join active users' ratings with movie genres
        ↓
Count genre frequency for each user
        ↓
Rank genres for each user
        ↓
Select the most frequently rated genre
        ↓
Store result into Cassandra
```

The following script was executed in the Hadoop/Spark virtual machine environment:

```bash
spark-submit /home/maria_dev/assignment02/scripts/task_3_favourite_genre.py
```

The Task 3 result is stored in Cassandra table:

```text
task3_user_favourite_genres
```

The result shows that 568 users rated at least 50 movies and received a favourite genre result.

<img src="screenshots/analysis_task_3.png" width="800">

The screenshot also validates that the Task 3 result was successfully stored into Cassandra. The number of inserted records is 568, and the count retrieved from the Cassandra result table `task3_user_favourite_genres` is also 568. This confirms that the analytical result was written and read back successfully.
<img src="screenshots/task_3_cassandra_storage.png" width="800">

---

### 8.3 Task 4 and Task 5: User Filtering Using CQL

Task 4 and Task 5 were completed using CQL because both tasks only require filtering records from the Cassandra `users` table.

Task 4 identifies users who are less than 20 years old. The result shows that there are 77 users younger than 20.

Task 5 identifies users whose occupation is `scientist` and whose age is between 30 and 40 years old. The result shows that there are 16 users matching this condition.

The analysis process is:

```text
Cassandra users table
        ↓
CQL filtering
        ↓
Task 4: age < 20
Task 5: occupation = scientist and age between 30 and 40
        ↓
Store filtered results into Cassandra result tables
```

The following script was executed in the virtual machine environment:

```bash
python /home/maria_dev/assignment02/scripts/task_4_5_cql_user_filters.py
```

The Task 4 result is stored in Cassandra table:

```text
task4_young_users
```

The Task 5 result is stored in Cassandra table:

```text
task5_scientist_users
```

Since `age` and `occupation` are not part of the primary key in the current Cassandra `users` table, `ALLOW FILTERING` was used for demonstration purposes. In a production Cassandra design, query-oriented tables or secondary indexes should be created for these access patterns.

<img src="screenshots/analysis_task_4_5.png" width="800">

---

### 8.4 Cassandra Result Table Validation

After the five analytical tasks were completed, the results were stored back into Cassandra as separate result tables. The result tables can be queried directly using CQL for validation.

The result tables are:

| Task   | Cassandra Result Table        | Expected Result                                |
| ------ | ----------------------------- | ---------------------------------------------- |
| Task 1 | `task1_average_movie_ratings` | Average rating and rating count for each movie |
| Task 2 | `task2_top_10_movies`         | Top 10 movies with the highest average ratings |
| Task 3 | `task3_user_favourite_genres` | Favourite genre for each active user           |
| Task 4 | `task4_young_users`           | Users younger than 20                          |
| Task 5 | `task5_scientist_users`       | Scientist users aged 30 to 40                  |

The following CQL commands can be used to validate the stored result tables:

```sql
USE movielens_ks;

SELECT COUNT(*) FROM task1_average_movie_ratings;
SELECT * FROM task1_average_movie_ratings LIMIT 10;

SELECT COUNT(*) FROM task2_top_10_movies;
SELECT * FROM task2_top_10_movies;

SELECT COUNT(*) FROM task3_user_favourite_genres;
SELECT * FROM task3_user_favourite_genres LIMIT 10;

SELECT COUNT(*) FROM task4_young_users;
SELECT * FROM task4_young_users LIMIT 10;

SELECT COUNT(*) FROM task5_scientist_users;
SELECT * FROM task5_scientist_users LIMIT 10;
```


#### task1
<img src="screenshots/task1_result.png" width="800">

#### task2
<img src="screenshots/task2_result.png" width="800">

#### task3
<img src="screenshots/task3_result.png" width="800">

#### task4
<img src="screenshots/task4_result.png" width="800">

#### task5
<img src="screenshots/task5_result.png" width="800">

The validation confirms that the analytical outputs were successfully persisted in Cassandra and can be retrieved later without rerunning the full analytical process.



## 9. Conclusion

This project built a complete MovieLens 100k data management pipeline using Apache Spark, Cassandra, and MongoDB. The raw dataset was first uploaded into HDFS and then loaded into Spark as RDDs. After that, the RDDs were parsed and transformed into Spark DataFrames for cleaning, preprocessing, database storage, and analytical tasks.

In this pipeline, Cassandra was used as the main structured NoSQL database. The cleaned user data and rating data were stored in Cassandra because both datasets have fixed schemas and are suitable for tabular storage. MongoDB was used to store movie information because one movie may belong to multiple genres, and MongoDB can naturally store the genre list as an array field inside a document.

After the processed data was stored, it was read back from Cassandra and MongoDB for analysis. Spark SQL was used for tasks that required joins, aggregation, ranking, and genre-level analysis, while CQL was used for simple filtering tasks on the Cassandra `users` table. Finally, the analytical results were stored back into Cassandra result tables and validated by querying the stored results.

### Task 1: Average Rating for Each Movie

The first task calculated the average rating for each movie by joining the rating records from Cassandra with movie information from MongoDB. The result included the movie ID, movie title, average rating, and rating count.

This task shows how rating records can be aggregated to generate movie-level analytical information. The rating count was also included because it provides important context. A movie with a high average rating but only a few ratings may not be as reliable as a movie with many ratings.

The final result was stored in the Cassandra table `task1_average_movie_ratings`.

### Task 2: Top 10 Movies by Average Rating

The second task identified the top 10 movies with the highest average ratings based on the result from Task 1. The result showed several movies with an average rating of 5.0.

However, some of these movies had very small rating counts, such as only one or two ratings. Therefore, although they ranked highly by average rating, the result should be interpreted carefully. This highlights the importance of considering both `average_rating` and `rating_count` when evaluating movie popularity or quality.

The final result was stored in the Cassandra table `task2_top_10_movies`.

### Task 3: Active Users and Favourite Movie Genre

The third task identified users who rated at least 50 movies and determined their favourite genre based on the genre they rated most frequently. This was the most complex analytical task in the project.

Since movie genres were stored in MongoDB as an array field, Spark temporarily exploded the genre array into multiple rows for analysis. Then, the rating data was joined with the movie genre data. For each active user, the number of ratings per genre was counted, and the most frequently rated genre was selected as the user's favourite genre.

The result showed that 568 users rated at least 50 movies and received a favourite genre result. Drama appeared frequently as a favourite genre, which may be related to the large number of Drama movies in the MovieLens dataset.

The final result was stored in the Cassandra table `task3_user_favourite_genres`.

### Task 4: Users Younger Than 20

The fourth task found all users who are less than 20 years old. Since this task only required filtering records from the Cassandra `users` table, it was completed using CQL.

The result showed that there were 77 users younger than 20. This task demonstrates how Cassandra can be used for direct filtering queries on structured user data.

Because `age` is not part of the primary key in the current Cassandra table design, `ALLOW FILTERING` was used for this query. This is acceptable for this assignment, but in a production Cassandra system, a query-oriented table or secondary index should be designed for this type of access pattern.

The final result was stored in the Cassandra table `task4_young_users`.

### Task 5: Scientist Users Aged 30 to 40

The fifth task found users whose occupation is `scientist` and whose age is between 30 and 40. This task was also completed using CQL because it only required filtering the Cassandra `users` table.

The result showed that 16 users matched this condition. This task further demonstrates the use of CQL for simple user filtering queries.

Similar to Task 4, `ALLOW FILTERING` was used because `occupation` and `age` are not part of the primary key. For a real-world Cassandra design, it would be better to create a table based on the query pattern, such as a table partitioned by occupation or age group.

The final result was stored in the Cassandra table `task5_scientist_users`.

### Overall Summary

Overall, this assignment successfully implemented a multi-database data pipeline. Spark was used as the main processing and analytical engine, Cassandra was used for structured data storage and final result storage, and MongoDB was used for semi-structured movie document storage.

The final Cassandra result tables are:

| Task   | Cassandra Result Table        | Number of Records |
| ------ | ----------------------------- | ----------------: |
| Task 1 | `task1_average_movie_ratings` |              1682 |
| Task 2 | `task2_top_10_movies`         |                10 |
| Task 3 | `task3_user_favourite_genres` |               568 |
| Task 4 | `task4_young_users`           |                77 |
| Task 5 | `task5_scientist_users`       |                16 |

The project demonstrates the full workflow of loading raw data, transforming it into structured DataFrames, storing data into NoSQL databases, reading data back for validation, performing analytical tasks, and storing final results for later retrieval.
